# Notes

# Import

# Исполнение

# Парадная версия

In [2]:
import os
import webbrowser
from getpass import getpass
from urllib.parse import parse_qs
from dotenv import load_dotenv
from requests_oauthlib import OAuth1Session

load_dotenv()
CLIENT_ID = os.getenv('ConsumerKey')
CLIENT_SECRET = os.getenv('ConsumerSecret')

def parse_token(text):
    data = parse_qs(text)
    return (data.get('oauth_token', [None])[0],
            data.get('oauth_token_secret', [None])[0])

session = OAuth1Session(CLIENT_ID, CLIENT_SECRET, callback_uri='oob', signature_type='BODY')
resp = session.post('https://authentication.fatsecret.com/oauth/request_token')
resp.raise_for_status()
request_token, request_token_secret = parse_token(resp.text)

auth_url = f'https://authentication.fatsecret.com/oauth/authorize?oauth_token={request_token}'
print(f'Перейдите по ссылке и разрешите доступ:\n{auth_url}')
webbrowser.open(auth_url)

verifier = getpass('Введите PIN-код: ').strip()

access_session = OAuth1Session(
    CLIENT_ID, CLIENT_SECRET,
    resource_owner_key=request_token,
    resource_owner_secret=request_token_secret,
    verifier=verifier,
    signature_type='BODY'
)
resp = access_session.post('https://authentication.fatsecret.com/oauth/access_token')
resp.raise_for_status()
access_token, access_token_secret = parse_token(resp.text)

api_session = OAuth1Session(
    CLIENT_ID, CLIENT_SECRET,
    resource_owner_key=access_token,
    resource_owner_secret=access_token_secret,
    signature_type='BODY'
)
response = api_session.get(
    'https://platform.fatsecret.com/rest/server.api',
    params={'method': 'profile.get', 'format': 'json'}
)
response.raise_for_status()
print('Данные профиля:')
print(response.json())

Перейдите по ссылке и разрешите доступ:
https://authentication.fatsecret.com/oauth/authorize?oauth_token=7dcd4f2f77c74cb6bcf989dc91744920


ValueError: GET/HEAD requests should not include body.